# FFTW C2C F90 MPI Parallel

- FFTW MPI F90 Interface http://fftw.org/doc/FFTW-MPI-Fortran-Interface.html#FFTW-MPI-Fortran-Interface

    - fftw_mpi_plan_dft_2d
    - fftw_mpi_plan_dft_3d

In [41]:
#=-----------------------------------------------------------------------=#

## 2D fftw_mpi_plan_dft_2d

In [1]:
%%writefile test01.f90
program main
    use, intrinsic :: iso_c_binding
    use MPI
    implicit none
    include 'fftw3-mpi.f03'
    integer :: mpirank, mpisize, mpierror
    integer(C_INTPTR_T) :: i, j, alloc_local, local_M, local_offset
    integer(C_INTPTR_T), parameter :: L = 3, M = 3
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:)

    call MPI_Init(mpierror)
    call MPI_Comm_rank(MPI_COMM_WORLD, mpirank, mpierror)
    call MPI_Comm_size(MPI_COMM_WORLD, mpisize, mpierror)
    call fftw_mpi_init()

!get local data size and allocate (note dimension reversal)
    alloc_local = fftw_mpi_local_size_2d(M, L, MPI_COMM_WORLD, &
                                       local_M, local_offset)
    cdata = fftw_alloc_complex(alloc_local)
    call c_f_pointer(cdata, data, [L,local_M])

!create MPI plan for in-place forward DFT (note dimension reversal)
    plan = fftw_mpi_plan_dft_2d(M, L, data, data, MPI_COMM_WORLD, &
                                  FFTW_FORWARD, FFTW_MEASURE)

!initialize data
    data = reshape([(.50,.0), (.73,.0), (.22,.0),  &
                    (.29,.0), (.65,.0), (.41,.0),  &
                    (.69,.0), (.25,.0), (.76,.0)], &
                    shape(data), order=[2, 1] )
    write(*,*) 'in:'; do j = 1, size(data,2);
        write(*, "(*(sf5.2spf5.2'j':x))") data(j,:); enddo;

!compute transform (as many times as desired)
    call fftw_mpi_execute_dft(plan, data, data)

    write(*,*) 'out:'; do j = 1, size(data,2);
        write(*, "(*(sf5.2spf5.2'j':x))") data(j,:); enddo;

    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
    call fftw_mpi_cleanup()
    call mpi_finalize(mpierror)
end

Writing test01.f90


In [2]:
%%bash
module load mpi/openmpi-x86_64
mpifort test01.f90 -fcheck=all -ffpe-trap= -w -fno-backtrace \
    -L $(pwd)/lo/lib -l fftw3_mpi -l fftw3 -l m -I $(pwd)/lo/include

In [3]:
%%bash
module load mpi/openmpi-x86_64
mpiexec -n 1 ./a.out

 in:
 0.50+0.00j  0.73+0.00j  0.22+0.00j
 0.29+0.00j  0.65+0.00j  0.41+0.00j
 0.69+0.00j  0.25+0.00j  0.76+0.00j
 out:
 4.50+0.00j -0.03-0.21j -0.03+0.21j
-0.07+0.30j -0.51-0.19j  0.61+0.93j
-0.07-0.30j  0.61-0.93j -0.51+0.19j


## 3D fftw_mpi_plan_dft_3d

In [4]:
%%writefile test02.f90
program main
    use, intrinsic :: iso_c_binding
    use MPI
    implicit none
    include 'fftw3-mpi.f03'
    integer :: mpirank, mpisize, mpierror
    integer(C_INTPTR_T), parameter :: L = 3, M = 3, N = 3
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:,:)
    integer(C_INTPTR_T) :: i, j, k, alloc_local, local_N, local_offset
    complex(C_DOUBLE_COMPLEX) :: s

    call MPI_Init(mpierror)
    call MPI_Comm_rank(MPI_COMM_WORLD, mpirank, mpierror)
    call MPI_Comm_size(MPI_COMM_WORLD, mpisize, mpierror)
    call fftw_mpi_init()

!get local data size and allocate (note dimension reversal)
    alloc_local = fftw_mpi_local_size_3d(N, M, L, MPI_COMM_WORLD, &
                                         local_N, local_offset)
    cdata = fftw_alloc_complex(alloc_local)
    call c_f_pointer(cdata, data, [L, M, local_N])

!create MPI plan for in-place forward DFT (note dimension reversal)
    plan = fftw_mpi_plan_dft_3d(N, M, L, data, data, MPI_COMM_WORLD, &
                                FFTW_FORWARD, FFTW_ESTIMATE)

!initialize data
    data = reshape([(.50,.0), (.73,.0), (.22,.0),  &
                    (.29,.0), (.65,.0), (.41,.0),  &
                    (.69,.0), (.25,.0), (.76,.0),  &
                    (.64,.0), (.60,.0), (.73,.0),  &
                    (.93,.0), (.24,.0), (.63,.0),  &
                    (.19,.0), (.73,.0), (.77,.0),  &
                    (.93,.0), (.70,.0), (.29,.0),  &
                    (.53,.0), (.34,.0), (.20,.0),  &
                    (.91,.0), (.20,.0), (.47,.0)], &
                    shape(data), order=[3, 2, 1] )
    write(*,*) 'in:'; do k = 1, size(data,1); do j = 1, size(data,2);
        write(*, "(*(sf5.2spf5.2'j':x))") data(k,j,:);
    enddo; write(*,"('--'g0'--')") k; enddo;

!compute transform (as many times as desired)
    call fftw_mpi_execute_dft(plan, data, data)

    write(*,*) 'out:'; do k = 1, size(data,1); do j = 1, size(data,2);
        write(*, "(*(sf5.2spf5.2'j':x))") data(k,j,:);
    enddo; write(*,"('--'g0'--')") k; enddo;
    s = sum(data)
    write(*, "('S: '*(sf8.4spf8.4'j':x))") s

    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
    call fftw_mpi_cleanup()
    call mpi_finalize(mpierror)
end

Writing test02.f90


In [6]:
%%bash
module load mpi/openmpi-x86_64
mpifort test02.f90 -fcheck=all -ffpe-trap= -w -fno-backtrace \
    -L $(pwd)/lo/lib -l fftw3_mpi -l fftw3 -l m -I $(pwd)/lo/include

In [7]:
%%bash
module load mpi/openmpi-x86_64
mpiexec -n 1 ./a.out

 in:
 0.50+0.00j  0.73+0.00j  0.22+0.00j
 0.29+0.00j  0.65+0.00j  0.41+0.00j
 0.69+0.00j  0.25+0.00j  0.76+0.00j
--1--
 0.64+0.00j  0.60+0.00j  0.73+0.00j
 0.93+0.00j  0.24+0.00j  0.63+0.00j
 0.19+0.00j  0.73+0.00j  0.77+0.00j
--2--
 0.93+0.00j  0.70+0.00j  0.29+0.00j
 0.53+0.00j  0.34+0.00j  0.20+0.00j
 0.91+0.00j  0.20+0.00j  0.47+0.00j
--3--
 out:
14.53+0.00j  1.15+0.03j  1.15-0.03j
 0.75+0.65j -0.53-1.32j  0.68+0.77j
 0.75-0.65j  0.68-0.77j -0.53+1.32j
--1--
-0.52-0.77j  0.01+0.85j -1.25+1.51j
-0.95+0.45j -1.24-0.11j -0.74+1.51j
-0.02+0.19j  1.90-0.50j  0.24-0.86j
--2--
-0.52+0.77j -1.25-1.51j  0.01-0.85j
-0.02-0.19j  0.24+0.86j  1.90+0.50j
-0.95-0.45j -0.74-1.51j -1.24+0.11j
--3--
S:  13.5000 +0.0000j


## 3D 12x12x12 random data 

In [20]:
%%writefile test03.f90
program main
    use, intrinsic :: iso_c_binding
    use MPI
    implicit none
    include 'fftw3-mpi.f03'
    integer :: mpirank, mpisize, mpierror
    integer(C_INTPTR_T), parameter :: L = 12, M = L, N = L
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:,:)
    integer(C_INTPTR_T) :: i, j, k, alloc_local, local_N, local_offset
    complex(C_DOUBLE_COMPLEX) :: s
    integer, allocatable, dimension(:) :: seed
    real :: r(L, M, N)

    call MPI_Init(mpierror)
    call MPI_Comm_rank(MPI_COMM_WORLD, mpirank, mpierror)
    call MPI_Comm_size(MPI_COMM_WORLD, mpisize, mpierror)
    call fftw_mpi_init()

!get local data size and allocate (note dimension reversal)

    alloc_local = fftw_mpi_local_size_3d(N, M, L, MPI_COMM_WORLD, &
                                         local_N, local_offset)
    cdata = fftw_alloc_complex(alloc_local)
    call c_f_pointer(cdata, data, [L, M, local_N])

!create MPI plan for in-place forward DFT (note dimension reversal)

    plan = fftw_mpi_plan_dft_3d(N, M, L, data, data, MPI_COMM_WORLD, &
                                FFTW_FORWARD, FFTW_ESTIMATE)

!initialize data

    seed = [.50, .73, .22, .29, .65, .41, .69, .25, 0., 0., 0., 0.]
    call random_seed(put=seed)
    call random_number(r)
    data = cmplx(r, 0)
    !write(*,*) 'in:'; do k = 1, size(data,1); do j = 1, size(data,2);
    !    write(*, "(*(sf5.2spf5.2'j':x))") data(k,j,:);
    !enddo; write(*,"('--'g0'--')") k; enddo;

!compute transform (as many times as desired)

    call fftw_mpi_execute_dft(plan, data, data)

    !write(*,*) 'out:'; do k = 1, size(data,1); do j = 1, size(data,2);
    !    write(*, "(*(sf6.2spf6.2'j':x))") data(k,j,:);
    !enddo; write(*,"('--'g0'--')") k; enddo;
    s = sum(data)
    write(*, "('S: '*(sf8.4spf8.4'j':x))") s

    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
    call fftw_mpi_cleanup()
    call mpi_finalize(mpierror)
end

Overwriting test03.f90


In [21]:
%%bash
module load mpi/openmpi-x86_64
mpifort test03.f90 -fcheck=all -ffpe-trap= -w -fno-backtrace \
    -L $(pwd)/lo/lib -l fftw3_mpi -l fftw3 -l m -I $(pwd)/lo/include

In [22]:
%%bash
module load mpi/openmpi-x86_64
mpiexec -n 1 ./a.out

S: 533.9813 +0.0000j


## processes > 1 and checks; random data 

In [13]:
%%writefile test04.f90
program main
    use, intrinsic :: iso_c_binding
    use MPI
    implicit none
    include 'fftw3-mpi.f03'
    integer :: mpirank, mpisize, mpierror
    integer(C_INTPTR_T), parameter :: L = 12, M = 12, N = 12
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:,:)
    integer(C_INTPTR_T) :: i, j, k, alloc_local, local_N, local_start
    complex(C_DOUBLE_COMPLEX) :: s, rs
    integer, allocatable :: seed(:)
    real, allocatable :: r(:,:,:), q(:,:,:)

    call MPI_Init(mpierror)
    call MPI_Comm_rank(MPI_COMM_WORLD, mpirank, mpierror)
    call MPI_Comm_size(MPI_COMM_WORLD, mpisize, mpierror)
    call fftw_mpi_init()

!get local data size and allocate (note dimension reversal)

    alloc_local = fftw_mpi_local_size_3d(N, M, L, MPI_COMM_WORLD, &
                                         local_N, local_start)
    cdata = fftw_alloc_complex(alloc_local)
    call c_f_pointer(cdata, data, [L, M, local_N])
    allocate(r(L, M, local_N), q(L, M, local_N))

!create MPI plan for in-place forward DFT (note dimension reversal)

    plan = fftw_mpi_plan_dft_3d(N, M, L, data, data, MPI_COMM_WORLD, &
                                FFTW_FORWARD, FFTW_ESTIMATE)

!initialize data

    seed = [.50, .73, .22, .29, .65, .41, .69, .25, 0., 0., 0., 0.]
    call random_seed(put=seed)
    call random_number(r)
    call random_number(q)
    data = cmplx(r, q)
    !write(*,*) 'in:'; do k = 1, size(data,1); do j = 1, size(data,2);
    !    write(*, "(*(sf5.2spf5.2'j':x))") data(k,j,:);
    !enddo; write(*,"('--'g0'--')") k; enddo;

!compute transform (as many times as desired)

    call fftw_mpi_execute_dft(plan, data, data)

    !write(*,*) 'out:'; do k = 1, size(data,1); do j = 1, size(data,2);
    !    write(*, "(*(sf6.2spf6.2'j':x))") data(k,j,:);
    !enddo; write(*,"('--'g0'--')") k; enddo;

!compute the checksum of processes

    s = sum(data)
    call MPI_Reduce(s,        &! send data
                    rs,       &! recv data
                    1,        &! count
                    MPI_DOUBLE_COMPLEX,  &! data type
                    MPI_SUM,  &! operation
                    0,        &! rank of root process
                    MPI_COMM_WORLD, mpierror)

!show result
    
    if (mpirank == 0) then
        write(*, "('S: '*(sf8.2spf8.2'j':x))") rs
        write(*,*)  &
    '   rank     l_N     l_s      shape(L, M,      N)      s' 
    !234567_1234567_1234567_1234567_1234567_1234567_1234567_
    endif
    write(*, '(*(g8.0))', advance='no')  &
        mpirank, local_N, local_start, shape(data)
    write(*, "(sf8.2spf8.2'j')") s
    
    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
    call fftw_mpi_cleanup()
    if (mpirank == 0) then
    endif
    call mpi_finalize(mpierror)
end

Overwriting test04.f90


In [14]:
%%bash
module load openmpi/gnu/2.0.4.14
mpifort test04.f90 -fcheck=all -ffpe-trap= -w -fno-backtrace \
    -L $(pwd)/lo/lib -l fftw3_mpi -l fftw3 -l m -I $(pwd)/lo/include

In [17]:
%%bash
module load openmpi/gnu/2.0.4.14
mpiexec -n 1 ./a.out

S:   533.98 +200.56j
    rank     l_N     l_s      shape(L, M,      N)      s
       0      12       0      12      12      12  533.98 +200.56j


In [18]:
%%bash
module load openmpi/gnu/2.0.4.14
mpiexec -n 2 ./a.out

S:   533.98  +20.56j
    rank     l_N     l_s      shape(L, M,      N)      s
       0       6       0      12      12       6 1211.87 +423.11j
       1       6       6      12      12       6 -677.89 -402.55j


In [19]:
%%bash
module load openmpi/gnu/2.0.4.14
mpiexec -n 4 ./a.out

S:   533.98+1461.19j
    rank     l_N     l_s      shape(L, M,      N)      s
       1       3       3      12      12       3   34.11  +52.28j
       2       3       6      12      12       3 -168.34 +170.23j
       3       3       9      12      12       3    0.00   +0.00j
       0       3       0      12      12       3  668.20+1238.68j


In [21]:
%%bash
module load openmpi/gnu/2.0.4.14
mpiexec -n 6 ./a.out

S:   533.98 +633.16j
    rank     l_N     l_s      shape(L, M,      N)      s
       1       2       2      12      12       2    0.00   +0.00j
       2       2       4      12      12       2    0.00   +0.00j
       3       2       6      12      12       2 -151.74 -414.01j
       4       2       8      12      12       2    0.00   +0.00j
       5       2      10      12      12       2    0.00   +0.00j
       0       2       0      12      12       2  685.73+1047.17j


## 3D initialize data to some function

- based on http://fftw.org/doc/FFTW-MPI-Fortran-Interface.html#FFTW-MPI-Fortran-Interface

Only check function x data, 4x4x4

In [11]:
%%writefile test06.f90
program main
    use, intrinsic :: iso_c_binding
    use MPI
    implicit none
    include 'fftw3-mpi.f03'
    integer :: mpirank, mpisize, mpierror
    integer(C_INTPTR_T), parameter :: L = 4, M = 4, N = 4
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:,:)
    integer(C_INTPTR_T) :: i, j, k, alloc_local, local_N, local_start
    complex(C_DOUBLE_COMPLEX) :: s, rs
    double precision :: r

    call MPI_Init(mpierror)
    call MPI_Comm_rank(MPI_COMM_WORLD, mpirank, mpierror)
    call MPI_Comm_size(MPI_COMM_WORLD, mpisize, mpierror)
    call fftw_mpi_init()

!get local data size and allocate (note dimension reversal)

    alloc_local = fftw_mpi_local_size_3d(N, M, L,  &
                     MPI_COMM_WORLD, local_N, local_start)
    cdata = fftw_alloc_complex(alloc_local)
    call c_f_pointer(cdata, data, [L, M, local_N])

!create MPI plan for in-place forward DFT (note dimension reversal)

    plan = fftw_mpi_plan_dft_3d(N, M, L, data, data,  &
                    MPI_COMM_WORLD, FFTW_FORWARD, FFTW_ESTIMATE)

!initialize data

if (mpirank == 0) then

    do k = 1, local_N
        do j = 1, M
            do i = 1, L
                r  = ((i-1)*M*L + (j-1)*M  + (k-1+local_start))
                data(i, j, k) = cmplx(r, 0)
                !data(i, j, k) = cmplx( sin( (               &
                !                (k + local_start) +         &
                !                (j-1) * M +                 &
                !                (i-1) * M * L ) / 1E3 ), 0 )
                !data(i, j, k) = ( (k + local_start) +  &
                !                 (j-1) * M +           &
                !                 (i-1) * M * L ) / 1E3
                !data(i, j, k) = cmplx( sin( sqrt(  &
                !    real(i)**2 + real(j)**2 +  &
                !    real(k + local_start)**2 ) ) , 0 )
!                write(*,'(*(g8.0))', advance='no') i, j, k
!                write(*, "(sf8.3spf8.3'j')") data(i,j,k)
            enddo
        enddo
    enddo
    write(*,"('in: 'g4.0'    S: 'f8.3spf8.3'j')") mpirank, sum(data)
    do k = 1, size(data,1); do j = 1, size(data,2)
        write(*, "(*(sf5.2spf5.2'j':x))") data(k,j,:)
    enddo; write(*,"('--'g0'--')") k; enddo

endif

!compute transform (as many times as desired)

!    call fftw_mpi_execute_dft(plan, data, data)

!compute the checksum of processes

!    s = sum(data)
!    call MPI_Reduce(s,        &! send data
!                    rs,       &! recv data
!                    1,        &! count
!                    MPI_DOUBLE_COMPLEX,  &! data type
!                    MPI_SUM,  &! operation
!                    0,        &! rank of root process
!                    MPI_COMM_WORLD, mpierror)

!show result
    
!    if (mpirank == 0) then
!        write(*, "('S: '*(sf8.2spf8.2'j':x))") rs
!        write(*,*)  &
!        '   rank     l_N     l_s      shape(L, M,      N)       s' 
!        !234567_1234567_1234567_1234567_1234567_1234567_ 1234567_
!    endif
!    write(*, '(*(g8.0))', advance='no')  &
!        mpirank, local_N, local_start, shape(data)
!    write(*, "(' 'sf8.2spf8.2'j')") s
    
    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
    call fftw_mpi_cleanup()
    call mpi_finalize(mpierror)
end

Overwriting test06.f90


In [12]:
%%bash
module load openmpi/gnu/2.0.4.14
mpifort test06.f90 -fcheck=all -ffpe-trap= -w -fno-backtrace \
    -L $(pwd)/lo/lib -l fftw3_mpi -l fftw3 -l m -I $(pwd)/lo/include

In [13]:
%%bash
module load openmpi/gnu/2.0.4.14
mpiexec -n 1 ./a.out

in:    0    S: 2016.000  +0.000j
 0.00+0.00j  1.00+0.00j  2.00+0.00j  3.00+0.00j
 4.00+0.00j  5.00+0.00j  6.00+0.00j  7.00+0.00j
 8.00+0.00j  9.00+0.00j 10.00+0.00j 11.00+0.00j
12.00+0.00j 13.00+0.00j 14.00+0.00j 15.00+0.00j
--1--
16.00+0.00j 17.00+0.00j 18.00+0.00j 19.00+0.00j
20.00+0.00j 21.00+0.00j 22.00+0.00j 23.00+0.00j
24.00+0.00j 25.00+0.00j 26.00+0.00j 27.00+0.00j
28.00+0.00j 29.00+0.00j 30.00+0.00j 31.00+0.00j
--2--
32.00+0.00j 33.00+0.00j 34.00+0.00j 35.00+0.00j
36.00+0.00j 37.00+0.00j 38.00+0.00j 39.00+0.00j
40.00+0.00j 41.00+0.00j 42.00+0.00j 43.00+0.00j
44.00+0.00j 45.00+0.00j 46.00+0.00j 47.00+0.00j
--3--
48.00+0.00j 49.00+0.00j 50.00+0.00j 51.00+0.00j
52.00+0.00j 53.00+0.00j 54.00+0.00j 55.00+0.00j
56.00+0.00j 57.00+0.00j 58.00+0.00j 59.00+0.00j
60.00+0.00j 61.00+0.00j 62.00+0.00j 63.00+0.00j
--4--


In [245]:
%%bash
module load openmpi/gnu/2.0.4.14
mpiexec -n 2 ./a.out

in:    0    S:    0.976  +0.000j
0.000+.000j 0.001+.000j
0.004+.000j 0.005+.000j
0.008+.000j 0.009+.000j
0.012+.000j 0.013+.000j
--1--
0.016+.000j 0.017+.000j
0.020+.000j 0.021+.000j
0.024+.000j 0.025+.000j
0.028+.000j 0.029+.000j
--2--
0.032+.000j 0.033+.000j
0.036+.000j 0.037+.000j
0.040+.000j 0.041+.000j
0.044+.000j 0.045+.000j
--3--
0.048+.000j 0.049+.000j
0.052+.000j 0.053+.000j
0.056+.000j 0.057+.000j
0.060+.000j 0.061+.000j
--4--


In [246]:
%%bash
module load openmpi/gnu/2.0.4.14
mpiexec -n 4 ./a.out

in:    0    S:    0.480  +0.000j
0.000+.000j
0.004+.000j
0.008+.000j
0.012+.000j
--1--
0.016+.000j
0.020+.000j
0.024+.000j
0.028+.000j
--2--
0.032+.000j
0.036+.000j
0.040+.000j
0.044+.000j
--3--
0.048+.000j
0.052+.000j
0.056+.000j
0.060+.000j
--4--


### 3D 12x12x12 function

In [250]:
%%writefile test05.f90
program main
    use, intrinsic :: iso_c_binding
    use MPI
    implicit none
    include 'fftw3-mpi.f03'
    integer :: mpirank, mpisize, mpierror
    integer(C_INTPTR_T), parameter :: L = 12, M = 12, N = 12
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:,:)
    integer(C_INTPTR_T) :: i, j, k, alloc_local, local_N, local_start
    complex(C_DOUBLE_COMPLEX) :: s, rs
    double precision :: r

    call MPI_Init(mpierror)
    call MPI_Comm_rank(MPI_COMM_WORLD, mpirank, mpierror)
    call MPI_Comm_size(MPI_COMM_WORLD, mpisize, mpierror)
    call fftw_mpi_init()

! get local data size and allocate (note dimension reversal)

    alloc_local = fftw_mpi_local_size_3d(N, M, L,  &
                     MPI_COMM_WORLD, local_N, local_start)
    cdata = fftw_alloc_complex(alloc_local)
    call c_f_pointer(cdata, data, [L, M, local_N])

! create MPI plan for in-place forward DFT (note dimension reversal)

    plan = fftw_mpi_plan_dft_3d(N, M, L, data, data,  &
                    MPI_COMM_WORLD, FFTW_FORWARD, FFTW_ESTIMATE)

! initialize data
! from fftw.org:  data(i, j) = my_function(i, j + local_j_offset)

    do k = 1, local_N
        do j = 1, M
            do i = 1, L
                r = ((i-1)*M*L + (j-1)*M  + (k-1+local_start))
                data(i, j, k) = cmplx(r, 0)
            enddo
        enddo
    enddo

! compute transform (as many times as desired)

    call fftw_mpi_execute_dft(plan, data, data)

! compute the checksum of processes

    s = sum(data)
    call MPI_Reduce(s,        &! send data
                    rs,       &! recv data
                    1,        &! count
                    MPI_DOUBLE_COMPLEX,  &! data type
                    MPI_SUM,  &! operation
                    0,        &! rank of root process
                    MPI_COMM_WORLD, mpierror)

! show result
    
    if (mpirank == 0) then
        write(*, "('S: '*(sf8.2spf8.2'j':x))") rs
        write(*,*)  &
        '   rank     l_N     l_s      shape(L, M,      N)       s' 
        !234567_1234567_1234567_1234567_1234567_1234567_ 1234567_
    endif
    write(*, '(*(g8.0))', advance='no')  &
        mpirank, local_N, local_start, shape(data)
    write(*, "(' 'sf8.2spf8.2'j')") s
    
    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
    call fftw_mpi_cleanup()
    call mpi_finalize(mpierror)
end

Overwriting test05.f90


In [251]:
%%bash
module load openmpi/gnu/2.0.4.14
mpifort test05.f90 -fcheck=all -ffpe-trap= -w -fno-backtrace \
    -L $(pwd)/lo/lib -l fftw3_mpi -l fftw3 -l m -I $(pwd)/lo/include

In [252]:
%%bash
module load openmpi/gnu/2.0.4.14
mpiexec -n 1 ./a.out

S:     0.00   -0.00j
    rank     l_N     l_s      shape(L, M,      N)       s
       0      12       0      12      12      12     0.00   -0.00j


In [253]:
%%bash
module load openmpi/gnu/2.0.4.14
mpiexec -n 2 ./a.out

S:     0.00   -0.00j
    rank     l_N     l_s      shape(L, M,      N)       s
       0       6       0      12      12       6  5184.00+6315.32j
       1       6       6      12      12       6 -5184.00-6315.32j


In [254]:
%%bash
module load openmpi/gnu/2.0.4.14
mpiexec -n 4 ./a.out

S:     0.00   -0.00j
    rank     l_N     l_s      shape(L, M,      N)       s
       0       3       0      12      12       3  7776.00+4720.98j
       1       3       3      12      12       3 -2592.00+1594.34j
       2       3       6      12      12       3 -2592.00 -730.34j
       3       3       9      12      12       3 -2592.00-5584.98j


In [255]:
%%bash
module load openmpi/gnu/2.0.4.14
mpiexec -n 6 ./a.out

S:     0.00   -0.00j
    rank     l_N     l_s      shape(L, M,      N)       s
       0       2       0      12      12       2  8640.00+3224.49j
       1       2       2      12      12       2 -1728.00+2360.49j
       2       2       4      12      12       2 -1728.00 +730.34j
       3       2       6      12      12       2 -1728.00 -231.51j
       4       2       8      12      12       2 -1728.00-1362.83j
       5       2      10      12      12       2 -1728.00-4720.98j


## 3D 500x500x500

In [274]:
%%writefile test10.f90
program main
    use, intrinsic :: iso_c_binding
    use MPI
    implicit none
    include 'fftw3-mpi.f03'
    integer :: mpirank, mpisize, mpierror
    integer(C_INTPTR_T), parameter :: L = 500, M = 500, N = 500
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:,:)
    integer(C_INTPTR_T) :: alloc_local, local_N, local_start
    complex(C_DOUBLE_COMPLEX) :: s, rs
    integer :: i, j, k
    double precision :: r, t1, t2

    call MPI_Init(mpierror)
    call MPI_Comm_rank(MPI_COMM_WORLD, mpirank, mpierror)
    call MPI_Comm_size(MPI_COMM_WORLD, mpisize, mpierror)
    if (mpirank == 0) then
        call cpu_time(t1)    ! <--- time measurement
    endif
        
    call fftw_mpi_init()

! get local data size and allocate (note dimension reversal)

    alloc_local = fftw_mpi_local_size_3d(N, M, L,  &
                     MPI_COMM_WORLD, local_N, local_start)
    cdata = fftw_alloc_complex(alloc_local)
    call c_f_pointer(cdata, data, [L, M, local_N])

! create MPI plan for in-place forward DFT (note dimension reversal)

    plan = fftw_mpi_plan_dft_3d(N, M, L, data, data, &
                    MPI_COMM_WORLD, FFTW_FORWARD, FFTW_ESTIMATE)

! initialize data
! from fftw.org:  data(i, j) = my_function(i, j + local_j_offset)

    do k = 1, local_N
        do j = 1, M
            do i = 1, L
                r = ((i-1)*M*L + (j-1)*M  + (k-1+local_start))
                data(i, j, k) = cmplx(r, 0)
            enddo
        enddo
    enddo

! compute transform (as many times as desired)

    call fftw_mpi_execute_dft(plan, data, data)

! compute the checksum of processes

    s = sum(data)
    call MPI_Reduce(s,        &! send data
                    rs,       &! recv data
                    1,        &! count
                    MPI_DOUBLE_COMPLEX,  &! data type
                    MPI_SUM,  &! operation
                    0,        &! rank of root process
                    MPI_COMM_WORLD, mpierror)
    
! show the result
    
    if (mpirank == 0) then
        call cpu_time(t2)    ! <--- time measurement        
        write(*, "('S: 'sf0.4spf0.4'j')", advance="no") rs
        write(*, "('    T: 'sf0.4' s')") t2-t1
        write(*,*)  &
        ' rank   l_N   l_s   shape            s' 
        !2345_12345_12345_12345_12345_12345_12345_
    endif
    write(*, '(*(g6.0))', advance='no')  &
        mpirank, local_N, local_start, shape(data)
    write(*, "('  'sf0.4spf0.4'j')") s
    
    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
    call fftw_mpi_cleanup()

    call mpi_finalize(mpierror)
end

Overwriting test10.f90


In [275]:
%%bash
module load openmpi/gnu/2.0.4.14
mpifort test10.f90 -fcheck=all -ffpe-trap= -w -fno-backtrace \
    -L $(pwd)/lo/lib -l fftw3_mpi -l fftw3 -l m -I $(pwd)/lo/include

In [276]:
%%bash
module load openmpi/gnu/2.0.4.14
mpiexec -n 1 ./a.out

S: .1225+1.0667j    T: 9.4131 s
  rank   l_N   l_s   shape            s
     0   500     0   500   500   500  .1225+1.0667j


In [268]:
%%bash
module load openmpi/gnu/2.0.4.14
mpiexec -n 2 ./a.out

S: .0008-.0035j    T: 6.8929 s
  rank   l_N   l_s   shape            s
     0   250     0   500   500   250  15624999.9971+56172660.3740j
     1   250   250   500   500   250  -15624999.9963-56172660.3775j


In [269]:
%%bash
module load openmpi/gnu/2.0.4.14
mpiexec -n 4 ./a.out

     3   125   375   500   500   125  -7812499.8118-52756445.9113j
     1   125   125   500   500   125  -7812500.1726+3478714.4719j
S: .0008-.0035j    T: 3.5594 s
  rank   l_N   l_s   shape            s
     0   125     0   500   500   125  23437500.1697+52693945.9020j
     2   125   250   500   500   125  -7812500.1845-3416214.4662j


In [270]:
%%bash
module load openmpi/gnu/2.0.4.14
mpiexec -n 6 ./a.out

     1    84    84   500   500    84  -5250000.0435+5475874.3342j
     3    84   252   500   500    84  -5250000.1546-1509784.3521j
     5    80   420   500   500    80  -4999999.7285-48964744.4737j
     2    84   168   500   500    84  -5250000.2077+1400477.5861j
     4    84   336   500   500    84  -5250000.0908-5697738.8406j
S: .0008-.0035j    T: 3.7213 s
  rank   l_N   l_s   shape            s
     0    84     0   500   500    84  26000000.2260+49295915.7425j


## Runs on an execution node

Compile using -O3 and uses system libraries

In [39]:
%%writefile test11.f90
program main
    use, intrinsic :: iso_c_binding
    use MPI
    implicit none
    include 'fftw3-mpi.f03'
    integer :: mpirank, mpisize, mpierror
    integer(C_INTPTR_T), parameter :: N = 500, M = N, L = N
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:,:)
    integer(C_INTPTR_T) :: alloc_local, local_N, local_start
    complex(C_DOUBLE_COMPLEX) :: s, rs
    integer :: i, j, k
    double precision :: r, t1, t2

    call MPI_Init(mpierror)
    call MPI_Comm_rank(MPI_COMM_WORLD, mpirank, mpierror)
    call MPI_Comm_size(MPI_COMM_WORLD, mpisize, mpierror)
    if (mpirank == 0) then
        call cpu_time(t1)    ! <--- time measurement
    endif

    call fftw_mpi_init()    

    ! get local data size and allocate (note dimension reversal)
    alloc_local = fftw_mpi_local_size_3d(N, M, L,  &
                 MPI_COMM_WORLD, local_N, local_start)
    cdata = fftw_alloc_complex(alloc_local)
    call c_f_pointer(cdata, data, [L, M, local_N])

    ! create MPI plan for in-place forward DFT (note dimension reversal)
    plan = fftw_mpi_plan_dft_3d(N, M, L, data, data,  &
                MPI_COMM_WORLD, FFTW_FORWARD, FFTW_ESTIMATE)

    ! initialize data
    if (mpirank == 0) then
      do k = 1, local_N
        do j = 1, M
          do i = 1, L
            r = ((i-1)*M*L + (j-1)*M  + (k + local_start))*1E-6
            data(i, j, k) = cmplx(r, 0)
          enddo
        enddo
      enddo
    endif

    ! compute transform (as many times as desired)
    call fftw_mpi_execute_dft(plan, data, data)

    ! compute the checksum of processes
    s = sum(data)
    call MPI_Reduce(s,        &! send data
                    rs,       &! recv data
                    1,        &! count
                    MPI_DOUBLE_COMPLEX,  &! data type
                    MPI_SUM,  &! operation
                    0,        &! rank of root process
                    MPI_COMM_WORLD, mpierror)
    
    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
    call fftw_mpi_cleanup()
    
    ! show the result
    if (mpirank == 0) then
        call cpu_time(t2)    ! <--- time measurement        
        write(*, "('S: 'sf0.4spf0.4'j')", advance="no") rs
        write(*, "('    T: 'sf0.4' s')", advance="no") t2-t1
        write(*, "('    N: 'g0)") mpisize
    endif
    
    call mpi_finalize(mpierror)
end

Overwriting test11.f90


- Using mathlibs/fftw/3.3.8_openmpi-3.1_gnu (loads fft & openmpi at same time)
- loaded before JupyterLab:
        module load anaconda3/2020.11
        conda activate /scratch/app/anaconda3/2020.11
        conda activate --stack $pwd/envs
        module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu

In [40]:
%%bash
dir=/scratch/app/mathlibs/fftw/3.3.8_openmpi-3.1_gnu
mpifort -O3 -o fc2cp test11.f90  \
    -L $dir/lib  -l fftw3_mpi  -l fftw3  -l m  -I $dir/include

Check

In [41]:
%%bash
mpiexec -n 1 ./fc2cp

S: 125.0000+.0000j    T: 7.8512 s    N: 1


In [42]:
%%bash
mpiexec -n 4 ./fc2cp

S: 125.0000-.0000j    T: 2.9339 s    N: 4


Copy to /scratch

In [43]:
%%bash
dst=/scratch${PWD#"/prj"}
mkdir -p $dst
cp fc2cp $dst

Configures the batch script 

In [1]:
%%writefile fc2cp.srm
#!/bin/bash
# 1,0 UA partitions:
#   cpu,       96 h,    21-50 nodes, 4/24  tasks
#   cpu_dev,   20 min., 1-4   nodes, 1/1   tasks
#   cpu_small, 72 h,    1-20  nodes, 16/96 tasks
#SBATCH -p cpu_small        # Select partition
#SBATCH --ntasks=1          # Total tasks
#SBATCH -J fc2cp            # Job name
#SBATCH --time=00:01:00     # Limit execution time
#SBATCH --exclusive         # Exclusive acccess to nodes

echo '========================================'
echo '- Job ID:' $SLURM_JOB_ID
echo '- Tasks per node:' $SLURM_NTASKS_PER_NODE
echo '- # of nodes in the job:' $SLURM_JOB_NUM_NODES
echo '- # of tasks:' $SLURM_NTASKS
echo '- Dir from which sbatch was invoked:' ${SLURM_SUBMIT_DIR##*/}
cd $SLURM_SUBMIT_DIR
echo -n '- List of nodes allocated to the job: '
nodeset -e $SLURM_JOB_NODELIST

# Executable config
EXEC=$PWD/fc2cp

# Modules config
echo '-- modules ----------------------------'
echo '$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu'
module load  mathlibs/fftw/3.3.8_openmpi-3.1_gnu

# Start
echo '$ srun --mpi=pmi2 -n' $SLURM_NTASKS ${EXEC##*/}
echo '-- output -----------------------------'
srun --mpi=pmi2 -n $SLURM_NTASKS $EXEC
echo '~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'

Overwriting fc2cp.srm


Submit batch job

### 1 of (1, 4, 9, 16, 36, 49, 64, 81)

In [45]:
! sbatch --ntasks=1 --partition=cpu_dev fc2cp.srm

Submitted batch job 862626


In [46]:
! squeue -n fc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID PARTITION ST       TIME  NODES
            862626   cpu_dev  R       0:00      1


In [48]:
! squeue -n fc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID PARTITION ST       TIME  NODES


In [49]:
! cat /scratch${PWD#"/prj"}/slurm-862626.out

- Job ID: 862626
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1243
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 1 fc2cp
-- output -----------------------------
S: 125.0000+.0000j    T: 9.0341 s    N: 1
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


#### 2nd and 3nd measurement

In [2]:
! sbatch --ntasks=1 fc2cp.srm
! sbatch --ntasks=1 fc2cp.srm

Submitted batch job 862679
Submitted batch job 862680


In [1]:
! squeue -n fc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            862679  cpu_small  PD  0:00     1    1
            862680  cpu_small  PD  0:00     1    1
            862696  cpu_small  PD  0:00     1    4
            862697  cpu_small  PD  0:00     1    4
            862698  cpu_small  PD  0:00     1    4
            862699  cpu_small  PD  0:00     1    9
            862700  cpu_small  PD  0:00     1    9
            862701  cpu_small  PD  0:00     1    9


In [1]:
! squeue -n fc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [2]:
! cat /scratch${PWD#"/prj"}/slurm-862679.out
! cat /scratch${PWD#"/prj"}/slurm-862680.out

- Job ID: 862679
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1318
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 1 fc2cp
-- output -----------------------------
S: 125.0000+.0000j    T: 9.0329 s    N: 1
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 862680
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 1 fc2cp
-- output -----------------------------
S: 125.0000+.0000j    T: 9.0998 s    N: 1
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


### 4 of (1, 4, 9, 16, 36, 49, 64, 81)

In [6]:
! sbatch --ntasks=4 fc2cp.srm
! sbatch --ntasks=4 fc2cp.srm
! sbatch --ntasks=4 fc2cp.srm

Submitted batch job 862696
Submitted batch job 862697
Submitted batch job 862698


In [7]:
! squeue -n fc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID PARTITION ST       TIME  NODES
            862679 cpu_small PD       0:00      1
            862680 cpu_small PD       0:00      1
            862696 cpu_small PD       0:00      1
            862697 cpu_small PD       0:00      1
            862698 cpu_small PD       0:00      1


In [3]:
! squeue -n fc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [4]:
! cat /scratch${PWD#"/prj"}/slurm-862696.out
! cat /scratch${PWD#"/prj"}/slurm-862697.out
! cat /scratch${PWD#"/prj"}/slurm-862698.out

- Job ID: 862696
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 4
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1301
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 4 fc2cp
-- output -----------------------------
S: 125.0000-.0000j    T: 2.8670 s    N: 4
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 862697
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 4
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 4 fc2cp
-- output -----------------------------
S: 125.0000-.0000j    T: 2.8545 s    N: 4
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 862698
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 4
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1301
-- modules 

### 9 of (1, 4, 9, 16, 36, 49, 64, 81)

In [8]:
! sbatch --ntasks=9 fc2cp.srm
! sbatch --ntasks=9 fc2cp.srm
! sbatch --ntasks=9 fc2cp.srm

Submitted batch job 862699
Submitted batch job 862700
Submitted batch job 862701


In [30]:
! squeue -n fc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            862679  cpu_small  PD  0:00     1    1
            862680  cpu_small  PD  0:00     1    1
            862696  cpu_small  PD  0:00     1    4
            862697  cpu_small  PD  0:00     1    4
            862698  cpu_small  PD  0:00     1    4
            862699  cpu_small  PD  0:00     1    9
            862700  cpu_small  PD  0:00     1    9
            862701  cpu_small  PD  0:00     1    9


In [3]:
! squeue --partition=cpu_small  -h -t pending,running -r | wc -l

538


In [4]:
! squeue -u $(whoami) -h -t pending,running -r | wc -l

19


In [10]:
! squeue --partition=cpu  -h -t pending,running -r | wc -l

17


In [11]:
! squeue --partition=cpu -o "%.18i  %.8u  %.2t %.5M %.5D %.4C"

             JOBID      USER  ST  TIME NODES CPUS
            853490  vitor.pa  PD  0:00    49 1176
            853818  thiago.m  PD  0:00    50 1200
            855489  victor.a  PD  0:00    50 1200
            856396  bruno.fa  PD  0:00    48 1152
            856761  gabriel.  PD  0:00    50 1200
            857816  thiago.m  PD  0:00    50 1200
            860332  larissa.  PD  0:00    50 1200
            852828  rafael.g  PD  0:00    30  720
            853649  tiago.ca  PD  0:00    32  544
            858053  lucas.fr  PD  0:00    24  576
            858056  lucas.fr  PD  0:00    24  576
            859965  kevin.co  PD  0:00    21  504
            861181  gabriela  PD  0:00    21  504
            862149  douglas.  PD  0:00    21    1
            772471  carla.sa  PD  0:00    21    1
            863159  leonardo  PD  0:00    32  768
            863301  leonardo  PD  0:00    32  768


In [1]:
! squeue --partition=cpu_dev -o "%.18i  %.8u  %.2t %.5M %.5D %.4C"

             JOBID      USER  ST  TIME NODES CPUS
            865408  fabio.ri   R 14:34     4   96


In [12]:
! squeue --partition=cpu_shared  -h -t pending,running -r | wc -l

158


In [5]:
! squeue -n fc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [6]:
! cat /scratch${PWD#"/prj"}/slurm-862699.out
! cat /scratch${PWD#"/prj"}/slurm-862700.out
! cat /scratch${PWD#"/prj"}/slurm-862701.out

- Job ID: 862699
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 9 fc2cp
-- output -----------------------------
S: 125.0000+.0000j    T: 3.3188 s    N: 9
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 862700
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1301
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 9 fc2cp
-- output -----------------------------
S: 125.0000+.0000j    T: 3.2935 s    N: 9
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 862701
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273
-- modules 

### 16 of (1, 4, 9, 16, 36, 49, 64, 81)

In [5]:
! sbatch --ntasks=16 fc2cp.srm
! sbatch --ntasks=16 fc2cp.srm
! sbatch --ntasks=16 fc2cp.srm

Submitted batch job 865583
Submitted batch job 865584
Submitted batch job 865585


In [7]:
! squeue -n fc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [8]:
! cat /scratch${PWD#"/prj"}/slurm-865583.out
! cat /scratch${PWD#"/prj"}/slurm-865584.out
! cat /scratch${PWD#"/prj"}/slurm-865585.out

- Job ID: 865583
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1301
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 16 fc2cp
-- output -----------------------------
S: 125.0000-.0000j    T: .9721 s    N: 16
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 865584
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 16 fc2cp
-- output -----------------------------
S: 125.0000-.0000j    T: .9640 s    N: 16
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 865585
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1301
-- mod

### 36 of (1, 4, 9, 16, 36, 49, 64, 81)

In [6]:
! sbatch --ntasks=36 fc2cp.srm
! sbatch --ntasks=36 fc2cp.srm
! sbatch --ntasks=36 fc2cp.srm

Submitted batch job 865587
Submitted batch job 865588
Submitted batch job 865589


In [9]:
! squeue -n fc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [10]:
! cat /scratch${PWD#"/prj"}/slurm-865587.out
! cat /scratch${PWD#"/prj"}/slurm-865588.out
! cat /scratch${PWD#"/prj"}/slurm-865589.out

- Job ID: 865587
- Tasks per node:
- # of nodes in the job: 2
- # of tasks: 36
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273 sdumont1301
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 36 fc2cp
-- output -----------------------------
S: 125.0000-.0000j    T: 1.2152 s    N: 36
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 865588
- Tasks per node:
- # of nodes in the job: 2
- # of tasks: 36
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273 sdumont1301
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 36 fc2cp
-- output -----------------------------
S: 125.0000-.0000j    T: 1.2307 s    N: 36
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 865589
- Tasks per node:
- # of nodes in the job: 2
- # of tasks: 36
- Dir from which sbatch was invoked: fft
- List of nodes allocated to t

### 49 of (1, 4, 9, 16, 36, 49, 64, 81)

In [7]:
! sbatch --ntasks=49 fc2cp.srm
! sbatch --ntasks=49 fc2cp.srm
! sbatch --ntasks=49 fc2cp.srm

Submitted batch job 865594
Submitted batch job 865595
Submitted batch job 865596


In [11]:
! squeue -n fc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [12]:
! cat /scratch${PWD#"/prj"}/slurm-865594.out
! cat /scratch${PWD#"/prj"}/slurm-865595.out
! cat /scratch${PWD#"/prj"}/slurm-865596.out

- Job ID: 865594
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 49
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273 sdumont1301 sdumont1321
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 49 fc2cp
-- output -----------------------------
S: 125.0000+.0000j    T: 1.1981 s    N: 49
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 865595
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 49
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273 sdumont1301 sdumont1321
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 49 fc2cp
-- output -----------------------------
S: 125.0000+.0000j    T: 1.3453 s    N: 49
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 865596
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 49
- Dir from which sbatch was invoked: fft
- List

### 64 of (1, 4, 9, 16, 36, 49, 64, 81)

In [8]:
! sbatch --ntasks=64 fc2cp.srm
! sbatch --ntasks=64 fc2cp.srm
! sbatch --ntasks=64 fc2cp.srm

Submitted batch job 865603
Submitted batch job 865604
Submitted batch job 865605


In [13]:
! squeue -n fc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [14]:
! cat /scratch${PWD#"/prj"}/slurm-865603.out
! cat /scratch${PWD#"/prj"}/slurm-865604.out
! cat /scratch${PWD#"/prj"}/slurm-865605.out

- Job ID: 865603
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 64
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273 sdumont1301 sdumont1321
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 64 fc2cp
-- output -----------------------------
S: 125.0000+.0000j    T: 1.2274 s    N: 64
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 865604
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 64
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273 sdumont1301 sdumont1321
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 64 fc2cp
-- output -----------------------------
S: 125.0000+.0000j    T: 1.7457 s    N: 64
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 865605
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 64
- Dir from which sbatch was invoked: fft
- List

### 81 of (1, 4, 9, 16, 36, 49, 64, 81)

In [9]:
! sbatch --ntasks=81 fc2cp.srm
! sbatch --ntasks=81 fc2cp.srm
! sbatch --ntasks=81 fc2cp.srm

Submitted batch job 865606
Submitted batch job 865607
Submitted batch job 865608


In [15]:
! squeue -n fc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [16]:
! cat /scratch${PWD#"/prj"}/slurm-865606.out
! cat /scratch${PWD#"/prj"}/slurm-865607.out
! cat /scratch${PWD#"/prj"}/slurm-865608.out

- Job ID: 865606
- Tasks per node:
- # of nodes in the job: 4
- # of tasks: 81
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1350 sdumont1351 sdumont1352 sdumont1353
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 81 fc2cp
-- output -----------------------------
S: 125.0000-.0000j    T: 1.4418 s    N: 81
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 865607
- Tasks per node:
- # of nodes in the job: 4
- # of tasks: 81
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1350 sdumont1351 sdumont1352 sdumont1353
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 81 fc2cp
-- output -----------------------------
S: 125.0000-.0000j    T: 1.6111 s    N: 81
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 865608
- Tasks per node:
- # of nodes in the job: 4
- # of tasks: 81
- Dir from which sbatch

## Including measurement of communication time

In [17]:
%%writefile fc2cp.f90
program main
    use, intrinsic :: iso_c_binding
    use MPI
    implicit none
    include 'fftw3-mpi.f03'
    integer :: mpirank, mpisize, mpierror
    integer(C_INTPTR_T), parameter :: N = 500, M = N, L = N
    type(C_PTR) :: plan, cdata
    complex(C_DOUBLE_COMPLEX), pointer :: data(:,:,:)
    integer(C_INTPTR_T) :: alloc_local, local_N, local_start
    complex(C_DOUBLE_COMPLEX) :: s, rs
    integer :: i, j, k
    double precision :: r, t0, t1, t2, t3

    call MPI_Init(mpierror)
    call MPI_Comm_rank(MPI_COMM_WORLD, mpirank, mpierror)
    call MPI_Comm_size(MPI_COMM_WORLD, mpisize, mpierror)
    if (mpirank == 0) then
        call cpu_time(t0)    ! <--- time measurement
    endif

    call fftw_mpi_init()    

    ! get local data size and allocate (note dimension reversal)
    alloc_local = fftw_mpi_local_size_3d(N, M, L,  &
                 MPI_COMM_WORLD, local_N, local_start)
    cdata = fftw_alloc_complex(alloc_local)
    call c_f_pointer(cdata, data, [L, M, local_N])

    ! create MPI plan for in-place forward DFT (note dimension reversal)
    plan = fftw_mpi_plan_dft_3d(N, M, L, data, data,  &
                MPI_COMM_WORLD, FFTW_FORWARD, FFTW_ESTIMATE)

    ! initialize data
    if (mpirank == 0) then
      do k = 1, local_N
        do j = 1, M
          do i = 1, L
            r = ((i-1)*M*L + (j-1)*M  + (k + local_start))*1E-6
            data(i, j, k) = cmplx(r, 0)
          enddo
        enddo
      enddo
    endif

    ! compute transform (as many times as desired)
    call fftw_mpi_execute_dft(plan, data, data)

    ! compute the checksum of processes
    s = sum(data)
    if (mpirank == 0) then
        call cpu_time(t2)    ! <--- time measurement
    endif
    call MPI_Reduce(s,        &! send data
                    rs,       &! recv data
                    1,        &! count
                    MPI_DOUBLE_COMPLEX,  &! data type
                    MPI_SUM,  &! operation
                    0,        &! rank of root process
                    MPI_COMM_WORLD, mpierror)
    if (mpirank == 0) then
        call cpu_time(t3)    ! <--- time measurement
    endif
    
    call fftw_destroy_plan(plan)
    call fftw_free(cdata)
    call fftw_mpi_cleanup()
    
    ! show the result
    if (mpirank == 0) then
        call cpu_time(t1)    ! <--- time measurement        
        write(*, "('S: 'sf0.4spf0.4'j')", advance="no") rs
        write(*, "('  T: 'sf0.4' s')", advance="no") t1-t0
        write(*, "('  Tc: 'sf0.4' s')", advance="no") t3-t2
        write(*, "('  N: 'g0)") mpisize
    endif
    
    call mpi_finalize(mpierror)
end

Writing fc2cp.f90


In [18]:
%%bash
module load  mathlibs/fftw/3.3.8_openmpi-3.1_gnu
dir=/scratch/app/mathlibs/fftw/3.3.8_openmpi-3.1_gnu
mpifort -O3 -o fc2cp fc2cp.f90  \
    -L $dir/lib  -l fftw3_mpi  -l fftw3  -l m  -I $dir/include

Check

In [19]:
%%bash
module load  mathlibs/fftw/3.3.8_openmpi-3.1_gnu
mpiexec -n 1 ./fc2cp

S: 125.0000+.0000j  T: 7.6017 s  Tc: .0001 s  N: 1


In [2]:
%%bash
module load  mathlibs/fftw/3.3.8_openmpi-3.1_gnu
mpiexec -n 24 ./fc2cp

S: 125.0000+.0000j  T: 2.4917 s  Tc: .0307 s  N: 24


Copy to /scratch

In [3]:
%%bash
dst=/scratch${PWD#"/prj"}
mkdir -p $dst
cp fc2cp $dst

Configures the batch script 

In [4]:
%%writefile fc2cp.srm
#!/bin/bash
# 1,0 UA partitions:
#   cpu,       96 h,    21-50 nodes, 4/24  tasks
#   cpu_dev,   20 min., 1-4   nodes, 1/1   tasks
#   cpu_small, 72 h,    1-20  nodes, 16/96 tasks
#SBATCH --partition cpu_small  # Select partition
#SBATCH --ntasks=1             # Total tasks
#SBATCH --job-name fc2cp       # Job name
#SBATCH --time=00:01:00        # Limit execution time
#SBATCH --exclusive            # Exclusive acccess to nodes
#   NTASKS: 1, 4, 9, 16, 36, 49, 64, 81
echo '========================================'
echo '- Job ID:' $SLURM_JOB_ID
echo '- Tasks per node:' $SLURM_NTASKS_PER_NODE
echo '- # of nodes in the job:' $SLURM_JOB_NUM_NODES
echo '- # of tasks:' $SLURM_NTASKS
echo '- Dir from which sbatch was invoked:' ${SLURM_SUBMIT_DIR##*/}
cd $SLURM_SUBMIT_DIR
echo -n '- List of nodes allocated to the job: '
nodeset -e $SLURM_JOB_NODELIST

# Executable config
EXEC=$PWD/fc2cp

# Modules config
echo '-- modules ----------------------------'
echo '$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu'
module load  mathlibs/fftw/3.3.8_openmpi-3.1_gnu

# Start
echo '$ srun --mpi=pmi2 -n' $SLURM_NTASKS ${EXEC##*/}
echo '-- output -----------------------------'
srun --mpi=pmi2 -n $SLURM_NTASKS $EXEC
echo '~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~'

Overwriting fc2cp.srm


Submit batch job

Check

In [5]:
! sbatch --partition=cpu_dev --ntasks=16 fc2cp.srm

Submitted batch job 868402


In [7]:
! squeue -n fc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C" --partition=cpu_dev

             JOBID  PARTITION  ST  TIME NODES CPUS


In [8]:
! cat /scratch${PWD#"/prj"}/slurm-868402.out

- Job ID: 868402
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1243
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 16 fc2cp
-- output -----------------------------
S: 125.0000-.0000j  T: .9686 s  Tc: .0039 s  N: 16
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


### 1 of (1, 4, 9, 16, 36, 49, 64, 81)

In [23]:
! sbatch --ntasks=1 fc2cp.srm
! sbatch --ntasks=1 fc2cp.srm
! sbatch --ntasks=1 fc2cp.srm

Submitted batch job 868223
Submitted batch job 868224
Submitted batch job 868225


In [25]:
! squeue --partition=cpu_small -h -t pending,running -r | wc -l

520


In [35]:
! squeue -n fc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS
            868223  cpu_small  PD  0:00     1    1
            868224  cpu_small  PD  0:00     1    1
            868225  cpu_small  PD  0:00     1    1
            868226  cpu_small  PD  0:00     1    4
            868227  cpu_small  PD  0:00     1    4
            868228  cpu_small  PD  0:00     1    4
            868229  cpu_small  PD  0:00     1    9
            868230  cpu_small  PD  0:00     1    9
            868231  cpu_small  PD  0:00     1    9
            868232  cpu_small  PD  0:00     1   16
            868233  cpu_small  PD  0:00     1   16
            868234  cpu_small  PD  0:00     1   16
            868235  cpu_small  PD  0:00     2   36
            868236  cpu_small  PD  0:00     2   36
            868237  cpu_small  PD  0:00     2   36
            868238  cpu_small  PD  0:00     3   49
            868239  cpu_small  PD  0:00     3   49
            868240  cpu_small  PD  0:00     3   49
            868241  cpu_small  

In [1]:
! squeue -n fc2cp -o "%.18i  %.9P  %.2t %.5M %.5D %.4C"

             JOBID  PARTITION  ST  TIME NODES CPUS


In [2]:
! cat /scratch${PWD#"/prj"}/slurm-868223.out
! cat /scratch${PWD#"/prj"}/slurm-868224.out
! cat /scratch${PWD#"/prj"}/slurm-868225.out

- Job ID: 868223
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1318
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 1 fc2cp
-- output -----------------------------
S: 125.0000+.0000j  T: 9.0295 s  Tc: .0001 s  N: 1
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868224
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1318
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 1 fc2cp
-- output -----------------------------
S: 125.0000+.0000j  T: 9.0085 s  Tc: .0001 s  N: 1
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868225
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 1
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumo

### 4 of (1, 4, 9, 16, 36, 49, 64, 81)

In [27]:
! sbatch --ntasks=4 fc2cp.srm
! sbatch --ntasks=4 fc2cp.srm
! sbatch --ntasks=4 fc2cp.srm

Submitted batch job 868226
Submitted batch job 868227
Submitted batch job 868228


In [3]:
! cat /scratch${PWD#"/prj"}/slurm-868226.out
! cat /scratch${PWD#"/prj"}/slurm-868227.out
! cat /scratch${PWD#"/prj"}/slurm-868228.out

- Job ID: 868226
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 4
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1318
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 4 fc2cp
-- output -----------------------------
S: 125.0000-.0000j  T: 2.8372 s  Tc: .0038 s  N: 4
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868227
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 4
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1318
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 4 fc2cp
-- output -----------------------------
S: 125.0000-.0000j  T: 2.8342 s  Tc: .0035 s  N: 4
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868228
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 4
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumo

### 9 of (1, 4, 9, 16, 36, 49, 64, 81)

In [29]:
! sbatch --ntasks=9 fc2cp.srm
! sbatch --ntasks=9 fc2cp.srm
! sbatch --ntasks=9 fc2cp.srm

Submitted batch job 868229
Submitted batch job 868230
Submitted batch job 868231


In [4]:
! cat /scratch${PWD#"/prj"}/slurm-868229.out
! cat /scratch${PWD#"/prj"}/slurm-868230.out
! cat /scratch${PWD#"/prj"}/slurm-868231.out

- Job ID: 868229
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1318
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 9 fc2cp
-- output -----------------------------
S: 125.0000+.0000j  T: 3.3230 s  Tc: .0448 s  N: 9
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868230
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1318
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 9 fc2cp
-- output -----------------------------
S: 125.0000+.0000j  T: 3.2635 s  Tc: .0172 s  N: 9
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868231
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 9
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumo

### 16 of (1, 4, 9, 16, 36, 49, 64, 81)

In [30]:
! sbatch --ntasks=16 fc2cp.srm
! sbatch --ntasks=16 fc2cp.srm
! sbatch --ntasks=16 fc2cp.srm

Submitted batch job 868232
Submitted batch job 868233
Submitted batch job 868234


In [5]:
! cat /scratch${PWD#"/prj"}/slurm-868232.out
! cat /scratch${PWD#"/prj"}/slurm-865584.out
! cat /scratch${PWD#"/prj"}/slurm-868233.out

- Job ID: 868232
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1319
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 16 fc2cp
-- output -----------------------------
S: 125.0000-.0000j  T: .9280 s  Tc: .0007 s  N: 16
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 865584
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1273
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 16 fc2cp
-- output -----------------------------
S: 125.0000-.0000j    T: .9640 s    N: 16
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868233
- Tasks per node:
- # of nodes in the job: 1
- # of tasks: 16
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont13

### 36 of (1, 4, 9, 16, 36, 49, 64, 81)

In [31]:
! sbatch --ntasks=36 fc2cp.srm
! sbatch --ntasks=36 fc2cp.srm
! sbatch --ntasks=36 fc2cp.srm

Submitted batch job 868235
Submitted batch job 868236
Submitted batch job 868237


In [6]:
! cat /scratch${PWD#"/prj"}/slurm-868235.out
! cat /scratch${PWD#"/prj"}/slurm-868236.out
! cat /scratch${PWD#"/prj"}/slurm-868237.out

- Job ID: 868235
- Tasks per node:
- # of nodes in the job: 2
- # of tasks: 36
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 36 fc2cp
-- output -----------------------------
S: 125.0000-.0000j  T: 1.1957 s  Tc: .0013 s  N: 36
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868236
- Tasks per node:
- # of nodes in the job: 2
- # of tasks: 36
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 36 fc2cp
-- output -----------------------------
S: 125.0000-.0000j  T: 1.4783 s  Tc: .0028 s  N: 36
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868237
- Tasks per node:
- # of nodes in the job: 2
- # of tasks: 36
- Dir from which sbatch was invoked: fft
- List of no

### 49 of (1, 4, 9, 16, 36, 49, 64, 81)

In [32]:
! sbatch --ntasks=49 fc2cp.srm
! sbatch --ntasks=49 fc2cp.srm
! sbatch --ntasks=49 fc2cp.srm

Submitted batch job 868238
Submitted batch job 868239
Submitted batch job 868240


In [2]:
! cat /scratch${PWD#"/prj"}/slurm-868238.out
! cat /scratch${PWD#"/prj"}/slurm-868239.out
! cat /scratch${PWD#"/prj"}/slurm-868240.out

- Job ID: 868238
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 49
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292 sdumont1337
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 49 fc2cp
-- output -----------------------------
S: 125.0000+.0000j  T: 1.8770 s  Tc: .0023 s  N: 49
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868239
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 49
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1338 sdumont1352 sdumont1353
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 49 fc2cp
-- output -----------------------------
S: 125.0000+.0000j  T: 1.2676 s  Tc: .0040 s  N: 49
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868240
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 49
- Dir from which sbatch was i

### 64 of (1, 4, 9, 16, 36, 49, 64, 81)

In [33]:
! sbatch --ntasks=64 fc2cp.srm
! sbatch --ntasks=64 fc2cp.srm
! sbatch --ntasks=64 fc2cp.srm

Submitted batch job 868241
Submitted batch job 868242
Submitted batch job 868243


In [3]:
! cat /scratch${PWD#"/prj"}/slurm-868241.out
! cat /scratch${PWD#"/prj"}/slurm-868242.out
! cat /scratch${PWD#"/prj"}/slurm-868243.out

- Job ID: 868241
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 64
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292 sdumont1353
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 64 fc2cp
-- output -----------------------------
S: 125.0000+.0000j  T: 1.2754 s  Tc: .0122 s  N: 64
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868242
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 64
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1350 sdumont1351 sdumont1352
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 64 fc2cp
-- output -----------------------------
S: 125.0000+.0000j  T: 1.2263 s  Tc: .0158 s  N: 64
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868243
- Tasks per node:
- # of nodes in the job: 3
- # of tasks: 64
- Dir from which sbatch was i

### 81 of (1, 4, 9, 16, 36, 49, 64, 81)

In [34]:
! sbatch --ntasks=81 fc2cp.srm
! sbatch --ntasks=81 fc2cp.srm
! sbatch --ntasks=81 fc2cp.srm

Submitted batch job 868244
Submitted batch job 868245
Submitted batch job 868246


In [4]:
! cat /scratch${PWD#"/prj"}/slurm-868244.out
! cat /scratch${PWD#"/prj"}/slurm-868245.out
! cat /scratch${PWD#"/prj"}/slurm-868246.out

- Job ID: 868244
- Tasks per node:
- # of nodes in the job: 4
- # of tasks: 81
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1350 sdumont1351 sdumont1352 sdumont1353
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 81 fc2cp
-- output -----------------------------
S: 125.0000-.0000j  T: 1.2679 s  Tc: .0035 s  N: 81
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868245
- Tasks per node:
- # of nodes in the job: 4
- # of tasks: 81
- Dir from which sbatch was invoked: fft
- List of nodes allocated to the job: sdumont1291 sdumont1292 sdumont1337 sdumont1338
-- modules ----------------------------
$ module load mathlibs/fftw/3.3.8_openmpi-3.1_gnu
$ srun --mpi=pmi2 -n 81 fc2cp
-- output -----------------------------
S: 125.0000-.0000j  T: 1.3913 s  Tc: .0001 s  N: 81
~~ end ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
- Job ID: 868246
- Tasks per node:
- # of nodes in the job: 4
- # of tasks: 81
- Dir

## Clean

In [8]:
%%bash
rm -f *.f90 *.srm fc2cp fc2cs a.out data.csv
rm -f /scratch${PWD#"/prj"}/slurm-*.out

## Version

In [11]:
%%bash
module load  mathlibs/fftw/3.3.8_openmpi-3.1_gnu
mpifort --version
ompi_info --version

GNU Fortran (GCC) 4.8.5 20150623 (Red Hat 4.8.5-36)
Copyright (C) 2015 Free Software Foundation, Inc.

GNU Fortran comes with NO WARRANTY, to the extent permitted by law.
You may redistribute copies of GNU Fortran
under the terms of the GNU General Public License.
For more information about these matters, see the file named COPYING

Open MPI v3.1.4

http://www.open-mpi.org/community/help/


In [2]:
! module avail 2>&1 | grep -i fftw

mathlibs/fftw/3.3.8_intel
mathlibs/fftw/3.3.8_openmpi-2.0_gnu
mathlibs/fftw/3.3.8_openmpi-2.0_intel
mathlibs/fftw/3.3.8_openmpi-3.1_gnu


In [16]:
! hostnamectl | grep Operating

  Operating System: Red Hat Enterprise Linux Server 7.6 (Maipo)


In [15]:
! lscpu

Architecture:          x86_64
CPU op-mode(s):        32-bit, 64-bit
Byte Order:            Little Endian
CPU(s):                24
On-line CPU(s) list:   0-23
Thread(s) per core:    1
Core(s) per socket:    12
Socket(s):             2
NUMA node(s):          2
Vendor ID:             GenuineIntel
CPU family:            6
Model:                 62
Model name:            Intel(R) Xeon(R) CPU E5-2695 v2 @ 2.40GHz
Stepping:              4
CPU MHz:               2667.773
CPU max MHz:           3200.0000
CPU min MHz:           1200.0000
BogoMIPS:              4799.91
Virtualization:        VT-x
L1d cache:             32K
L1i cache:             32K
L2 cache:              256K
L3 cache:              30720K
NUMA node0 CPU(s):     0-11
NUMA node1 CPU(s):     12-23
Flags:                 fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush dts acpi mmx fxsr sse sse2 ss ht tm pbe syscall nx pdpe1gb rdtscp lm constant_tsc arch_perfmon pebs bts rep_good nopl xtopology nonsto